# SmartHire — Job Recommendation
TF-IDF + cosine similarity produces a Top-N ranked list.

In [1]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 60)
print("SMART HIRE - JOB RECOMMENDATION SYSTEM")
print("=" * 60)

print("Libraries imported successfully!")

SMART HIRE - JOB RECOMMENDATION SYSTEM
Libraries imported successfully!


In [2]:
# ============================================================
# CELL 2: LOAD NAUKRI JOB DATASET
# ============================================================

jobs_df = pd.read_csv(
    "../data/raw/naukri_com-job_sample.csv"
)

print("=" * 60)
print("NAUKRI JOB DATASET LOADED")
print("=" * 60)

print("Dataset Shape:", jobs_df.shape)

print("\nColumns:")
print(jobs_df.columns.tolist())

print("\nFirst 5 Rows:")
display(jobs_df.head())

NAUKRI JOB DATASET LOADED
Dataset Shape: (22000, 14)

Columns:
['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']

First 5 Rows:


,company,education,experience,industry,jobdescription,jobid,joblocation_address,jobtitle,numberofpositions,payrate,postdate,site_name,skills,uniq_id
0,MM Media Pvt Ltd,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,0 - 1 yrs,Media / Entertainment / Internet,Job Description Send me Jobs like this Quali...,210516002263,Chennai,Walkin Data Entry Operator (night Shift),NaN,"1,50,000 - 2,25,000 P.A",2016-05-21 19:30:00 +0000,NaN,ITES,43b19632647068535437c774b6ca6cf8
1,find live infotech,UG: B.Tech/B.E. - Any Specialization PG:MBA/PG...,0 - 0 yrs,Advertising / PR / MR / Event Management,Job Description Send me Jobs like this Quali...,210516002391,Chennai,Work Based Onhome Based Part Time.,60.0,"1,50,000 - 2,50,000 P.A. 20000",2016-05-21 19:30:00 +0000,NaN,Marketing,d4c72325e57f89f364812b5ed5a795f0
2,Softtech Career Infosystem Pvt. Ltd,UG: Any Graduate - Any Specialization PG:Any P...,4 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this - as ...,101016900534,Bengaluru,Pl/sql Developer - SQL,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,c47df6f4cfdf5b46f1fd713ba61b9eba
3,Onboard HRServices LLP,UG: Any Graduate - Any Specialization PG:CA Do...,11 - 15 yrs,Banking / Financial Services / Broking,Job Description Send me Jobs like this - Inv...,81016900536,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Manager/ad/partner - Indirect Tax - CA,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,Accounts,115d28f140f694dd1cc61c53d03c66ae
4,Spire Technologies and Solutions Pvt. Ltd.,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,6 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this Pleas...,120916002122,Bengaluru,JAVA Technical Lead (6-8 yrs) -,4.0,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,a12553fc03bc7bcced8b1bb8963f97b4


In [3]:
# ============================================================
# CELL 3: SELECT IMPORTANT COLUMNS
# ============================================================

jobs_df = jobs_df[
    [
        "company",
        "experience",
        "industry",
        "jobdescription",
        "joblocation_address",
        "jobtitle",
        "numberofpositions",
        "payrate",
        "postdate",
        "skills"
    ]
].copy()

print("=" * 60)
print("IMPORTANT JOB COLUMNS SELECTED")
print("=" * 60)

print("Shape:", jobs_df.shape)

print("\nColumns:")
print(jobs_df.columns.tolist())

IMPORTANT JOB COLUMNS SELECTED
Shape: (22000, 10)

Columns:
['company', 'experience', 'industry', 'jobdescription', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'skills']


In [4]:
# ============================================================
# CELL 4: CHECK MISSING VALUES
# ============================================================

print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

missing_values = jobs_df.isnull().sum()

print(missing_values)

MISSING VALUE ANALYSIS
company                    4
experience                 4
industry                   5
jobdescription             4
joblocation_address      501
jobtitle                   0
numberofpositions      17536
payrate                   97
postdate                  23
skills                   528
dtype: int64


In [5]:
# ============================================================
# CELL 5: CLEAN TEXT COLUMNS
# ============================================================

text_columns = [
    "company",
    "experience",
    "industry",
    "jobdescription",
    "joblocation_address",
    "jobtitle",
    "skills"
]

for column in text_columns:
    jobs_df[column] = (
        jobs_df[column]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

print("=" * 60)
print("TEXT CLEANING COMPLETED")
print("=" * 60)

print("Missing values after cleaning:")

print(
    jobs_df[text_columns]
    .isnull()
    .sum()
)

TEXT CLEANING COMPLETED
Missing values after cleaning:
company                0
experience             0
industry               0
jobdescription         0
joblocation_address    0
jobtitle               0
skills                 0
dtype: int64


In [6]:
# ============================================================
# CELL 6: CREATE COMBINED JOB TEXT
# ============================================================

jobs_df["job_text"] = (
    jobs_df["jobtitle"] + " " +
    jobs_df["skills"] + " " +
    jobs_df["jobdescription"] + " " +
    jobs_df["industry"] + " " +
    jobs_df["experience"]
)

print("=" * 60)
print("COMBINED JOB TEXT CREATED")
print("=" * 60)

print("\nExample Job Text:")
print(jobs_df["job_text"].iloc[0])

COMBINED JOB TEXT CREATED

Example Job Text:
walkin data entry operator (night shift) ites job description   send me jobs like this qualifications: - == > 10th to graduation & any skill: - == > basic computer knowledge job requirement : - == > system or laptop type of job: - == > full time or part time languages : - == > tamil & english. experience : - == > freshers & experience payment details: - 1 form per day 5/- 10 form per day 50/- 100 form per day 500/- monthly you can earn 15000/- per month selection process: - == > easy selection process,so what are you waiting for? apply now & grab best opportunity to make your carrier & to improve your earing skills. more detail contact mr hari 8678902528 9003010282 salary:inr 1,50,000 - 2,25,000 p.a industry: media / entertainment / internet functional area: ites , bpo , kpo , lpo , customer service , operations role category:other role:fresher keyskills english typing part time data entry selection process desired candidate profile educatio

In [7]:
# ============================================================
# CELL 7: REMOVE DUPLICATE JOBS
# ============================================================

before = len(jobs_df)

jobs_df = jobs_df.drop_duplicates(
    subset=["jobtitle", "company", "jobdescription"]
)

after = len(jobs_df)

print("=" * 60)
print("DUPLICATE REMOVAL")
print("=" * 60)

print("Rows before:", before)
print("Rows after :", after)
print("Duplicates removed:", before - after)

DUPLICATE REMOVAL
Rows before: 22000
Rows after : 21851
Duplicates removed: 149


In [8]:
# ============================================================
# CELL 8: FINAL JOB DATASET CHECK
# ============================================================

print("=" * 60)
print("FINAL JOB DATASET")
print("=" * 60)

print("Shape:", jobs_df.shape)

print("\nColumns:")
print(jobs_df.columns.tolist())

print("\nSample Jobs:")

display(
    jobs_df[
        [
            "jobtitle",
            "company",
            "joblocation_address",
            "skills",
            "experience"
        ]
    ].head(10)
)

FINAL JOB DATASET
Shape: (21851, 11)

Columns:
['company', 'experience', 'industry', 'jobdescription', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'skills', 'job_text']

Sample Jobs:


,jobtitle,company,joblocation_address,skills,experience
0,walkin data entry operator (night shift),mm media pvt ltd,chennai,ites,0 - 1 yrs
1,work based onhome based part time.,find live infotech,chennai,marketing,0 - 0 yrs
2,pl/sql developer - sql,softtech career infosystem pvt. ltd,bengaluru,it software - application programming,4 - 8 yrs
3,manager/ad/partner - indirect tax - ca,onboard hrservices llp,"mumbai, bengaluru, kolkata, chennai, coimbator...",accounts,11 - 15 yrs
4,java technical lead (6-8 yrs) -,spire technologies and solutions pvt. ltd.,bengaluru,it software - application programming,6 - 8 yrs
5,walk in - as400 developer - pfsweb global serv...,pfs web global services pvt ltd,bengaluru,it software - application programming,2 - 5 yrs
6,php developer,kinesis management consultant pvt. ltd,"delhi ncr, mumbai, bengaluru, kochi, greater n...",it software - application programming,1 - 3 yrs
7,member technical staff-wire harness/cable harn...,agile hr consultancy pvt. ltd. hiring for ross...,bengaluru,production,2 - 7 yrs
8,team leader,hansum india electronics pvt.ltd.,bengaluru,production,1 - 3 yrs
9,german translator,accenture,bengaluru,ites,1 - 5 yrs


In [9]:
# ============================================================
# CELL 9: TF-IDF VECTORIZATION FOR JOBS
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000
)

job_tfidf_matrix = tfidf_vectorizer.fit_transform(
    jobs_df["job_text"]
)

print("=" * 60)
print("TF-IDF VECTORIZATION COMPLETED")
print("=" * 60)

print("Number of Jobs:", len(jobs_df))
print("TF-IDF Matrix Shape:", job_tfidf_matrix.shape)
print("Number of Features:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF VECTORIZATION COMPLETED
Number of Jobs: 21851
TF-IDF Matrix Shape: (21851, 50000)
Number of Features: 50000


In [10]:
# ============================================================
# CELL 10: JOB RECOMMENDATION FUNCTION
# ============================================================

def recommend_jobs(resume_text, top_n=5):

    # Convert resume text into TF-IDF
    resume_vector = tfidf_vectorizer.transform(
        [resume_text]
    )

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        resume_vector,
        job_tfidf_matrix
    ).flatten()

    # Get indices of top matching jobs
    top_indices = similarity_scores.argsort()[
        ::-1
    ][:top_n]

    # Create recommendation dataframe
    recommendations = jobs_df.iloc[
        top_indices
    ].copy()

    # Add match score
    recommendations["match_score"] = (
        similarity_scores[top_indices] * 100
    )

    return recommendations

In [11]:
# ============================================================
# CELL 11: TEST JOB RECOMMENDATION
# ============================================================

sample_resume = """
Computer Science engineering student with skills in
Python, Java, SQL, Machine Learning, Data Analysis,
Pandas, NumPy, Scikit-learn, Natural Language Processing,
TensorFlow, Git and REST APIs.

Experience in developing machine learning projects,
data preprocessing, model training, classification,
recommendation systems and Python applications.
"""

recommendations = recommend_jobs(
    sample_resume,
    top_n=5
)

print("=" * 60)
print("TOP 5 JOB RECOMMENDATIONS")
print("=" * 60)

display(
    recommendations[
        [
            "jobtitle",
            "company",
            "joblocation_address",
            "experience",
            "skills",
            "match_score"
        ]
    ]
)

TOP 5 JOB RECOMMENDATIONS


,jobtitle,company,joblocation_address,experience,skills,match_score
21993,developer engineer and data scientist,pin click - startup,bengaluru,3 - 4 yrs,analytics & business intelligence,41.145132
10637,data scientist machine learning,brillio technologies pvt. ltd,bengaluru/bangalore,4 - 7 yrs,analytics & business intelligence,39.215171
17442,software engineer - machine learning,quora,mumbai,2 - 5 yrs,it software - telecom software,38.306494
10864,sr scientist/ nlp/ machine learning/ data mini...,career maker,bengaluru/bangalore,2 - 7 yrs,analytics & business intelligence,37.795824
15561,machine learning scientist,career maker,bengaluru/bangalore,4 - 9 yrs,it software - embedded,36.082248


In [12]:
# ============================================================
# CELL 12: DISPLAY RECOMMENDATIONS
# ============================================================

print("=" * 60)
print("SMART HIRE - RECOMMENDED JOBS")
print("=" * 60)

for i, (_, job) in enumerate(
    recommendations.iterrows(),
    start=1
):

    print(f"\n{i}. {job['jobtitle']}")
    print(f"   Company    : {job['company']}")
    print(f"   Location   : {job['joblocation_address']}")
    print(f"   Experience : {job['experience']}")
    print(f"   Skills     : {job['skills']}")
    print(
        f"   Match Score: "
        f"{job['match_score']:.2f}%"
    )

SMART HIRE - RECOMMENDED JOBS

1. developer engineer and data scientist
   Company    : pin click - startup
   Location   : bengaluru
   Experience : 3 - 4 yrs
   Skills     : analytics & business intelligence
   Match Score: 41.15%

2. data scientist machine learning
   Company    : brillio technologies pvt. ltd
   Location   : bengaluru/bangalore
   Experience : 4 - 7 yrs
   Skills     : analytics & business intelligence
   Match Score: 39.22%

3. software engineer - machine learning
   Company    : quora
   Location   : mumbai
   Experience : 2 - 5 yrs
   Skills     : it software - telecom software
   Match Score: 38.31%

4. sr scientist/ nlp/ machine learning/ data mining professional
   Company    : career maker
   Location   : bengaluru/bangalore
   Experience : 2 - 7 yrs
   Skills     : analytics & business intelligence
   Match Score: 37.80%

5. machine learning scientist
   Company    : career maker
   Location   : bengaluru/bangalore
   Experience : 4 - 9 yrs
   Skills     : 

In [13]:
# ============================================================
# CELL 13: SAVE CLEAN JOB DATASET
# ============================================================

output_path = "../data/processed/jobs_clean.csv"

jobs_df.to_csv(
    output_path,
    index=False
)

print("=" * 60)
print("CLEAN JOB DATASET SAVED")
print("=" * 60)

print("File:", output_path)
print("Rows:", len(jobs_df))
print("Columns:", len(jobs_df.columns))

CLEAN JOB DATASET SAVED
File: ../data/processed/jobs_clean.csv
Rows: 21851
Columns: 11


In [14]:
# ============================================================
# CELL 14: SAVE JOB TF-IDF VECTORIZER
# ============================================================

import joblib

joblib.dump(
    tfidf_vectorizer,
    "../models/job_tfidf_vectorizer.pkl"
)

joblib.dump(
    job_tfidf_matrix,
    "../models/job_tfidf_matrix.pkl"
)

print("=" * 60)
print("JOB RECOMMENDER MODELS SAVED")
print("=" * 60)

print("1. models/job_tfidf_vectorizer.pkl")
print("2. models/job_tfidf_matrix.pkl")

JOB RECOMMENDER MODELS SAVED
1. models/job_tfidf_vectorizer.pkl
2. models/job_tfidf_matrix.pkl


In [15]:
# ============================================================
# CELL 15: SKILL GAP ANALYSIS - SETUP
# ============================================================

import re

# Common technical and professional skills
SKILL_LIST = [
    "python",
    "java",
    "c++",
    "c",
    "javascript",
    "typescript",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "html",
    "css",
    "react",
    "angular",
    "node.js",
    "django",
    "flask",
    "spring boot",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data science",
    "data analysis",
    "natural language processing",
    "nlp",
    "computer vision",
    "tensorflow",
    "keras",
    "pytorch",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "power bi",
    "tableau",
    "excel",
    "aws",
    "azure",
    "google cloud",
    "docker",
    "kubernetes",
    "git",
    "github",
    "linux",
    "rest api",
    "api",
    "spark",
    "hadoop",
    "communication",
    "leadership",
    "problem solving"
]

print("=" * 60)
print("SKILL GAP ANALYSIS")
print("=" * 60)

print("Total skills in skill dictionary:", len(SKILL_LIST))

SKILL GAP ANALYSIS
Total skills in skill dictionary: 52


In [16]:
# ============================================================
# CELL 16: EXTRACT SKILLS FROM RESUME
# ============================================================

def extract_skills(text):

    text = str(text).lower()

    found_skills = []

    for skill in SKILL_LIST:

        # Escape special characters
        pattern = r"\b" + re.escape(skill) + r"\b"

        if re.search(pattern, text):
            found_skills.append(skill)

    return sorted(set(found_skills))


resume_skills = extract_skills(sample_resume)

print("=" * 60)
print("RESUME SKILLS")
print("=" * 60)

for skill in resume_skills:
    print("✓", skill)

RESUME SKILLS
✓ data analysis
✓ git
✓ java
✓ machine learning
✓ natural language processing
✓ numpy
✓ pandas
✓ python
✓ scikit-learn
✓ sql
✓ tensorflow


In [17]:
# ============================================================
# CELL 17: EXTRACT REQUIRED SKILLS
# ============================================================

best_job = recommendations.iloc[0]

job_text = (
    str(best_job["jobtitle"]) + " " +
    str(best_job["skills"]) + " " +
    str(best_job["jobdescription"])
)

required_skills = extract_skills(job_text)

print("=" * 60)
print("JOB REQUIRED SKILLS")
print("=" * 60)

for skill in required_skills:
    print("•", skill)

JOB REQUIRED SKILLS
• data science
• hadoop
• java
• javascript
• machine learning
• natural language processing
• python
• spark


In [18]:
# ============================================================
# CELL 18: CALCULATE SKILL GAP
# ============================================================

missing_skills = sorted(
    set(required_skills) - set(resume_skills)
)

matching_skills = sorted(
    set(required_skills) & set(resume_skills)
)

print("=" * 60)
print("SKILL GAP ANALYSIS RESULT")
print("=" * 60)

print("\nMatching Skills:")
for skill in matching_skills:
    print("✓", skill)

print("\nMissing Skills:")
for skill in missing_skills:
    print("✗", skill)

print("\nTotal Required Skills:", len(required_skills))
print("Matching Skills:", len(matching_skills))
print("Missing Skills:", len(missing_skills))

SKILL GAP ANALYSIS RESULT

Matching Skills:
✓ java
✓ machine learning
✓ natural language processing
✓ python

Missing Skills:
✗ data science
✗ hadoop
✗ javascript
✗ spark

Total Required Skills: 8
Matching Skills: 4
Missing Skills: 4


In [19]:
# ============================================================
# CELL 19: SKILL MATCH SCORE
# ============================================================

if len(required_skills) > 0:

    skill_match_score = (
        len(matching_skills)
        / len(required_skills)
    ) * 100

else:

    skill_match_score = 0


print("=" * 60)
print("SKILL MATCH SCORE")
print("=" * 60)

print(
    f"Skill Match: "
    f"{skill_match_score:.2f}%"
)

SKILL MATCH SCORE
Skill Match: 50.00%


In [20]:
# ============================================================
# CELL 20: COMPLETE SKILL GAP REPORT
# ============================================================

print("=" * 60)
print("SMART HIRE - SKILL GAP REPORT")
print("=" * 60)

skill_gap_report = []

for _, job in recommendations.iterrows():

    job_text = (
        str(job["jobtitle"]) + " " +
        str(job["skills"]) + " " +
        str(job["jobdescription"])
    )

    required_skills = extract_skills(job_text)

    matching_skills = sorted(
        set(required_skills) & set(resume_skills)
    )

    missing_skills = sorted(
        set(required_skills) - set(resume_skills)
    )

    if len(required_skills) > 0:
        skill_score = (
            len(matching_skills) /
            len(required_skills)
        ) * 100
    else:
        skill_score = 0

    skill_gap_report.append({
        "Job Title": job["jobtitle"],
        "Company": job["company"],
        "Match Score": round(
            job["match_score"], 2
        ),
        "Skill Match": round(
            skill_score, 2
        ),
        "Matching Skills": ", ".join(
            matching_skills
        ),
        "Missing Skills": ", ".join(
            missing_skills
        )
    })

skill_gap_df = pd.DataFrame(
    skill_gap_report
)

display(skill_gap_df)

SMART HIRE - SKILL GAP REPORT


,Job Title,Company,Match Score,Skill Match,Matching Skills,Missing Skills
0,developer engineer and data scientist,pin click - startup,41.15,50.00,"java, machine learning, natural language proce...","data science, hadoop, javascript, spark"
1,data scientist machine learning,brillio technologies pvt. ltd,39.22,54.55,"java, machine learning, natural language proce...","artificial intelligence, c, hadoop, linux, spark"
2,software engineer - machine learning,quora,38.31,66.67,"machine learning, python",c
3,sr scientist/ nlp/ machine learning/ data mini...,career maker,37.80,50.00,"data analysis, java, machine learning, python,...","c, communication, linux, nlp, problem solving"
4,machine learning scientist,career maker,36.08,50.00,"java, machine learning, python","c, communication, problem solving"


In [21]:
# ============================================================
# CELL 21: SAVE SKILL GAP REPORT
# ============================================================

skill_gap_df.to_csv(
    "../reports/skill_gap_report.csv",
    index=False
)

print("=" * 60)
print("SKILL GAP REPORT SAVED")
print("=" * 60)

print(
    "File: ../reports/skill_gap_report.csv"
)

SKILL GAP REPORT SAVED
File: ../reports/skill_gap_report.csv
